# MEI Tools in Google Colab

This notebook walks through the complete workflow for extracting, editing, and updating metadata in a corpus of MEI files, using Google Colab as the environment.

The workflow has three stages:

1. **Extract** — scan a folder of MEI files and write a CSV of their current metadata
2. **Edit** — review and correct the CSV (in Google Sheets, Excel, or any text editor)
3. **Update** — apply the corrected CSV back to every MEI file in the corpus

---

## Before You Begin

You need:

- A Google account with Google Drive
- A folder of MEI files (any mix of MuseScore, Sibelius, Humdrum/Verovio, or mei-friend exports) uploaded to Drive
- This notebook open in Colab

---

## Step 1 — Install MEI Tools

> **Note:** To test a development branch, replace `@main` with the branch name, e.g. `@dev-26`.

In [ ]:
!pip install git+https://github.com/RichardFreedman/mei_tools.git@main

---

## Step 2 — Mount Drive and Set Up Folders

Run this single setup cell. Set **one path** — your project folder on Google Drive. All five workflow folders (A through E) are created inside it automatically.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ── Set this to your project folder in Google Drive ──
# All five workflow folders (A through E) will be created inside it.
project_folder = '/content/drive/MyDrive/my_mei_project'

folders = {
    'A_mei_to_process':            os.path.join(project_folder, 'A_mei_to_process'),
    'B_extracted_metadata_csv':    os.path.join(project_folder, 'B_extracted_metadata_csv'),
    'C_updated_metadata_csv':      os.path.join(project_folder, 'C_updated_metadata_csv'),
    'D_mei_with_updated_metadata': os.path.join(project_folder, 'D_mei_with_updated_metadata'),
    'E_mei_with_updated_music_features':   os.path.join(project_folder, 'E_mei_with_updated_music_features'),
}

for name, path in folders.items():
    os.makedirs(path, exist_ok=True)
    print(f'{name:35s}  →  {path}')


For a project at `MyDrive/my_mei_project/` this produces:

```
A_mei_to_process                   →  /content/drive/MyDrive/my_mei_project/A_mei_to_process
B_extracted_metadata_csv           →  /content/drive/MyDrive/my_mei_project/B_extracted_metadata_csv
C_updated_metadata_csv             →  /content/drive/MyDrive/my_mei_project/C_updated_metadata_csv
D_mei_with_updated_metadata        →  /content/drive/MyDrive/my_mei_project/D_mei_with_updated_metadata
E_mei_with_updated_music_features          →  /content/drive/MyDrive/my_mei_project/E_mei_with_updated_music_features
```

Upload your MEI files to `A_mei_to_process`. All five folders are available as `folders['A_mei_to_process']`, `folders['B_extracted_metadata_csv']`, etc. for the remaining steps.


---

## Step 3 — Extract Metadata to CSV

In [ ]:
from mei_tools import MEI_Metadata_Extractor

extractor = MEI_Metadata_Extractor(verbose=True)
extractor.save_csvs(
    input_folder=folders['A_mei_to_process'],
    output_folder=folders['B_extracted_metadata_csv']
)


### What this produces

One CSV file is written for each source type found in your corpus:

| Source type | CSV filename |
|---|---|
| MuseScore | `muse_score_extracted_metadata.csv` |
| Sibelius | `sib_extracted_metadata.csv` |
| Humdrum / Verovio | `hum_drum_extracted_metadata.csv` |
| mei-friend | `mei_friend_extracted_metadata.csv` |

If all your files come from the same application you will get just one CSV.

### CSV columns

| Column | What it contains |
|---|---|
| `filename` | basename of the .mei file — **do not edit this** |
| `source_type` | auto-detected application type |
| `mei_version` | MEI schema version from the file |
| `title` | main title (fileDesc/titleStmt) |
| `title_subordinate` | subordinate/movement titles, pipe-separated (mei-friend) |
| `composer_name` | composer's name |
| `composer_auth` | authority system, e.g. `VIAF` or `GND` |
| `composer_auth_uri` | full URI, e.g. `https://viaf.org/viaf/12304462/` |
| `composer_codedval` | coded value within authority (GND) |
| `editors` | pipe-separated list of `Name [role]` entries, e.g. `Jane Smith [editor]\|Bob Jones [encoder]` |
| `encoding_date` | date of encoding, ISO format |
| `rights` | rights/copyright statement |
| `publisher` | publisher name (MuseScore) |
| `distributor` | distributor(s), pipe-separated (MuseScore) |
| `genre` | genre term from workList |
| `encoding_application` | application name(s) and version(s) |
| `work_title` | title from workList/work |
| `movement_name` | movement name, Humdrum OMD field |
| `source_title` | title of the physical source (Humdrum) |
| `source_composer` | composer as recorded in sourceDesc (Humdrum) |
| `source_editor` | editor(s) from EED field (Humdrum) |
| `source_encoder` | encoder(s) from ENC field (Humdrum) |
| `edition_version` | edition version, EEV field (Humdrum) |
| `encoding_annot` | encoding annotation, ONB field (Humdrum) |
| `humdrum_id` | value of the `!!!id` reference key (Humdrum) |

**Editing rules:**

- Leave any cell **blank** to keep the existing MEI content unchanged — only non-empty cells are applied.
- For fields that accept multiple values (`editors`, `distributor`), separate entries with a pipe character: `Name One [role]|Name Two [role]`
- The `filename` and `source_type` columns are used to match rows to files — do not change them.

---

## Step 4 — Edit the CSV

Open the CSV from your Drive in **Google Sheets**, **Excel**, or any text editor. Fill in or correct the metadata values, then save it back to Drive (keep the same filename or update the path in Step 5).

### Using Google Sheets as a live metadata source

If you prefer to maintain metadata in a Google Sheet:

1. Upload the CSV to Google Sheets (`File → Import`)
2. When ready, publish it as CSV: `File → Share → Publish to web → select the sheet → CSV`
3. Copy the published URL — you can pass this directly to the updater in Step 5 instead of a file path

---

## Step 5 — Apply Updated Metadata to MEI Files

Choose **one** of the three options below depending on where your edited CSV lives.

### Option A — from a local Drive file

Copy your edited CSV into `folders['C_updated_metadata_csv']`. Adjust the filename for the source type you updated (e.g. `hum_drum_extracted_metadata.csv`, `muse_score_extracted_metadata.csv`, `sib_extracted_metadata.csv`, or `mei_friend_extracted_metadata.csv`).


In [ ]:
from mei_tools import MEI_Metadata_Updater_Generic
import os

csv_source = os.path.join(folders['C_updated_metadata_csv'], 'hum_drum_extracted_metadata.csv')

updater = MEI_Metadata_Updater_Generic(verbose=True)
updater.process_folder(
    input_folder=folders['A_mei_to_process'],
    csv_source=csv_source,
    output_folder=folders['D_mei_with_updated_metadata']
)


### Option B — from a Google Sheets published URL

In [ ]:
from mei_tools import MEI_Metadata_Updater_Generic

csv_source = 'https://docs.google.com/spreadsheets/d/e/YOURKEY/pub?output=csv'

updater = MEI_Metadata_Updater_Generic(verbose=True)
updater.process_folder(
    input_folder=folders['A_mei_to_process'],
    csv_source=csv_source,
    output_folder=folders['D_mei_with_updated_metadata']
)


### Option C — from a raw GitHub URL

In [ ]:
from mei_tools import MEI_Metadata_Updater_Generic

csv_source = 'https://raw.githubusercontent.com/yourorg/yourrepo/main/metadata/hum_drum_metadata.csv'

updater = MEI_Metadata_Updater_Generic(verbose=True)
updater.process_folder(
    input_folder=folders['A_mei_to_process'],
    csv_source=csv_source,
    output_folder=folders['D_mei_with_updated_metadata']
)


### Output

Each matching MEI file is written to `folders['D_mei_with_updated_metadata']` with `_rev` appended to the filename, e.g. `Ror0101_rev.mei`. The original files in `folders['A_mei_to_process']` are never modified.


---

## Step 6 (Optional) — CRIM Project Workflow

If you are working with the **CRIM (Citations: The Renaissance Imitation Mass)** project, use `crim_mode=True`. This expects the CRIM column schema (`MEI_Name`, `Title`, `Composer_VIAF`, `Editor`, etc.) and delegates processing to the full CRIM metadata writer, which includes the `manifestationList`.

In [ ]:
import os
from mei_tools import MEI_Metadata_Extractor, MEI_Metadata_Updater_Generic

# Extract using CRIM column names — produces crim_extracted_metadata.csv
extractor = MEI_Metadata_Extractor(verbose=True, crim_mode=True)
extractor.save_csvs(
    input_folder=folders['A_mei_to_process'],
    output_folder=folders['B_extracted_metadata_csv']
)

# (Edit the CSV, then apply)
updater = MEI_Metadata_Updater_Generic(verbose=True)
updater.process_folder(
    input_folder=folders['A_mei_to_process'],
    csv_source=os.path.join(folders['C_updated_metadata_csv'], 'crim_extracted_metadata.csv'),
    output_folder=folders['D_mei_with_updated_metadata'],
    crim_mode=True
)


---

## Step 7 (Optional) — Music Feature Corrections

After updating metadata, you can apply music feature corrections to the revised files in `folders['D_mei_with_updated_metadata']`. Corrected files are written to `folders['E_mei_with_updated_music_features']`. Edit the `True` / `False` values to control which corrections are applied. See the [README](https://github.com/RichardFreedman/mei_tools/blob/main/README.md) for a full description of each parameter.


In [ ]:
from mei_tools import MEI_Music_Feature_Processor
import glob

music_feature_processor = MEI_Music_Feature_Processor()

for mei_path in sorted(glob.glob(folders['D_mei_with_updated_metadata'] + '/*.mei')):
    music_feature_processor.process_music_features(
        mei_path,
        folders['E_mei_with_updated_music_features'],
        # --- most common corrections (True = apply) ---
        resolve_multibar_ties=True,
        remove_incipit=True,
        remove_pb=True,
        remove_sb=True,
        remove_annotation=True,
        remove_ligature_bracket=True,
        remove_dir=True,
        remove_variants=True,
        remove_anchored_text=True,
        remove_timestamp=True,
        remove_chord=True,
        check_for_chords=True,
        fix_elisions=True,
        fix_musescore_elisions=False,
        slur_to_tie=True,
        correct_ficta=True,
        voice_labels=True,
        correct_mrests=True,
        # --- less common corrections (False = skip) ---
        remove_senfl_bracket=False,
        remove_empty_verse=False,
        remove_lyrics=False,
        collapse_layers=False,
        correct_cmme_time_signatures=False,
        correct_jrp_time_signatures=False,
        remove_incipit_leuven=False,
    )


---

## Troubleshooting

### "No .mei files found"
Check that `folders['A_mei_to_process']` points to the correct Drive location and that the `.mei` files are directly in that folder (not in subfolders). Re-run the setup cell and confirm the printed paths are correct.

### Changes not appearing after code edits
If you modify mei_tools code locally and re-install, restart the Colab runtime (`Runtime → Restart runtime`) before re-running cells to clear the cached module.

### Composer not appearing in titleStmt
Make sure the `composer_name` column in your CSV is filled in. Blank cells are intentionally ignored. The updater writes the composer as `fileDesc/titleStmt/persName[@role='composer']` (or inside a `respStmt` for MuseScore, Sibelius, and mei-friend files).

### Wrong encoding detected
If a file is classified as the wrong source type, you can override detection by setting the `source_type` column in the CSV to one of: `musescore`, `sibelius`, `humdrum`, `mei_friend`.
